In [1]:
import polars as pl
from pybiomart import Server
import os
import numpy as np
import pandas as pd

In [2]:
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2'
DATASET_DIR = '/group/pmc021/amunif/epi-thesis/dataset'
HISTONE_DIR = os.path.join(WORKING_DIR, 'dataset', 'histone_overlap')

# Dataset Loading

## Load preprocessed HepG2 data

In [3]:
# Schema for gene data
schema_genes = pl.Schema({
    "chromosome"                : pl.String(),
    "tss -2kb"                  : pl.Int64(),
    "tss +2kb"                  : pl.Int64(),
    "test_id"                   : pl.String(),
    "gene_id"                   : pl.String(),
    "Chromosome/scaffold name"  : pl.String(),
    "Gene start (bp)"           : pl.Int64(),
    "Gene end (bp)"             : pl.Int64(),
    "Strand"                    : pl.Int64(),
    "Gene name"                 : pl.String(),
    "Gene stable ID"            : pl.String(),
    "Gene type"                 : pl.String(),
    "tss"                       : pl.Int64(),
    "locus"                     : pl.String(),
    "orig_start"                : pl.Int64(),
    "orig_end"                  : pl.Int64()
})

In [4]:
# Read the gene data
gene_pl = pl.read_csv(os.path.join(WORKING_DIR, "dataset", "ensembl_top1.csv"),
                     schema=schema_genes,
                     has_header=False,
                     separator="\t")

In [5]:
# Get the gene list
gene_list = gene_pl.select(pl.col('gene_id')).to_numpy().flatten()
len(gene_list)

22154

## Load Overlap Histone Data

In [6]:
# Schema for overlap histone
schema_histone_overlap = pl.Schema({
    "chromosome"                : pl.String(),
    "tss -2kb"                  : pl.Int64(),
    "tss +2kb"                  : pl.Int64(),
    "test_id"                   : pl.String(),
    "gene_id"                   : pl.String(),
    "Chromosome/scaffold name"  : pl.String(),
    "Gene start (bp)"           : pl.Int64(),
    "Gene end (bp)"             : pl.Int64(),
    "Strand"                    : pl.Int64(),
    "Gene name"                 : pl.String(),
    "Gene stable ID"            : pl.String(),
    "Gene type"                 : pl.String(),
    "tss"                       : pl.Int64(),
    "locus"                     : pl.String(),
    "orig_start"                : pl.Int64(),
    "orig_end"                  : pl.Int64(),
    "histone_chr"               : pl.String(),
    "histone_start"             : pl.Int64(),
    "histone_end"               : pl.Int64(),
    "histone_name"              : pl.String()
})

# Building Matrix Data

In [7]:
def build_matrix(gene_list, histone_name):
    # Open histone overlap file
    histone_pl = pl.read_csv(os.path.join(HISTONE_DIR, f'ensembl_top1_{histone_name}.bed'), 
                         schema = schema_histone_overlap, 
                         separator="\t",
                         has_header=False)

    # Create partitions to optimize searching
    histone_partitions = histone_pl.partition_by("gene_id", as_dict=True)

    # Build histone matrix
    histone_list = []
    i = 1
    for gene in gene_list:
        histone_arr = np.zeros(4000, dtype=float)
        count = 0
        
        if((gene,) in histone_partitions):
            # print(f"{gene} -  EXIST")
            
            partition = histone_partitions[(gene,)].to_numpy()
            count = partition.shape[0]
    
            for row in partition:
                start = row[1]
                idx_start = row[17] - start
                idx_end = row[18] - start
                # print(f"{row[19]}: {idx_start} - {idx_end}")
                
                histone_arr[idx_start: idx_end] = 1.0
        # else:
        #     print(f"{gene} -  NOT EXIST")
    
        # print(f"SUM: {np.sum(histone_arr)}, COUNT: {count}")
        entry = [gene, histone_arr, count]
    
        # print(f"{i}/{len(gene_list)}: {gene}, count: {count}")
    
        histone_list.append(entry)
        i += 1

    # Convert the array into Numpy arrays
    histone_np = np.array(histone_list, dtype="object")

    # Convert to Pandas dataframe
    histone_pd = pd.DataFrame(histone_np, columns=["gene_id", f"{histone_name}", f"{histone_name}_count"])

    # Convert to Polars dataframe
    histone_pl = pl.from_pandas(histone_pd)

    return histone_pl

## Build Matrix

In [8]:
# histone_names = ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']

H3K4me3_pl = build_matrix(gene_list, 'H3K4me3')
H3K9ac_pl  = build_matrix(gene_list, 'H3K9ac')
H3K9me3_pl = build_matrix(gene_list, 'H3K9me3')
H3K27ac_pl = build_matrix(gene_list, 'H3K27ac')
H3K27me3_pl = build_matrix(gene_list, 'H3K27me3')

In [9]:
H3K4me3_pl

gene_id,H3K4me3,H3K4me3_count
str,list[f64],i64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6
…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0


In [10]:
H3K9ac_pl

gene_id,H3K9ac,H3K9ac_count
str,list[f64],i64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0
"""XLOC_000008""","[0.0, 0.0, … 0.0]",3
…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0


In [11]:
H3K9me3_pl

gene_id,H3K9me3,H3K9me3_count
str,list[f64],i64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0
"""XLOC_000008""","[0.0, 0.0, … 0.0]",0
…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0


In [12]:
H3K27ac_pl

gene_id,H3K27ac,H3K27ac_count
str,list[f64],i64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0
"""XLOC_000008""","[0.0, 0.0, … 0.0]",2
…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0


In [13]:
H3K27me3_pl

gene_id,H3K27me3,H3K27me3_count
str,list[f64],i64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0
"""XLOC_000008""","[0.0, 0.0, … 0.0]",0
…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0


## Join into single matrix

In [14]:
# Join into single Polars dataframe
gene_w_histone = H3K4me3_pl \
                    .join(H3K9ac_pl, on='gene_id') \
                    .join(H3K9me3_pl, on='gene_id') \
                    .join(H3K27ac_pl, on='gene_id') \
                    .join(H3K27me3_pl, on='gene_id')

In [15]:
gene_w_histone

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0
…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0


# Join histone matrix with HepG2

## Load HepG2 data

In [16]:
label_column = "value_1"

In [17]:
# Load HepG2 expression dataset
hepg2_pl = pl.read_csv(os.path.join(DATASET_DIR, 'GSM3718064_HepG2_exp.txt'), separator="\t")

In [18]:
hepg2_pl

test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant
str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str
"""XLOC_000001""","""XLOC_000001""","""OR4F5""","""chr1:69090-70008""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no"""
"""XLOC_000002""","""XLOC_000002""","""LOC100132062,LOC100133331""","""chr1:323891-328581""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0473925,0.0341591,-0.472389,0.0,1.0,1.0,"""no"""
"""XLOC_000003""","""XLOC_000003""","""OR4F29""","""chr1:367658-368597""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no"""
"""XLOC_000004""","""XLOC_000004""","""LOC643837""","""chr1:761585-794889""","""hepg2_hr2""","""hepg2_hr3""","""OK""",2.46857,2.8058,0.184736,0.455074,0.63585,0.999565,"""no"""
"""XLOC_000005""","""XLOC_000005""","""-""","""chr1:840263-843900""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030028""","""XLOC_030028""","""-""","""chrY:13319431-13324829""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.337879,0.457897,0.438516,1.13793,0.25685,0.999565,"""no"""
"""XLOC_030029""","""XLOC_030029""","""-""","""chrY:13325034-13326120""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.396888,0.357658,-0.150148,0.0,1.0,1.0,"""no"""
"""XLOC_030030""","""XLOC_030030""","""-""","""chrY:13331089-13332358""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.477227,0.127331,-1.90609,-1.86192,0.1249,0.999565,"""no"""


In [19]:
hepg2_pl = hepg2_pl.select(["gene_id", label_column])

In [20]:
hepg2_pl

gene_id,value_1
str,f64
"""XLOC_000001""",0.0
"""XLOC_000002""",0.0473925
"""XLOC_000003""",0.0
"""XLOC_000004""",2.46857
"""XLOC_000005""",0.0
…,…
"""XLOC_030028""",0.337879
"""XLOC_030029""",0.396888
"""XLOC_030030""",0.477227


## Join with histone dataset

In [21]:
# Join with histone dataset
gene_w_label = gene_w_histone.join(hepg2_pl, on='gene_id')

In [22]:
gene_w_label

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,f64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0888452
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,4.04743
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0,26.7934
…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0


## Create a label column based on median value

In [23]:
# Create a label column based on median value
value_1_median = gene_w_label[label_column].median()
print(value_1_median)

1.1420949999999999


In [26]:
gene_w_label = gene_w_label.with_columns(pl.when(gene_w_label[label_column] >= value_1_median)
                    .then(1)
                    .otherwise(0)
                    .alias('label'))

In [27]:
gene_w_label

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1,label
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,f64,i32
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0888452,0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,4.04743,1
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0,26.7934,1
…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0


In [28]:
# Check the label distribution
gene_w_label.group_by("label").len()

label,len
i32,u32
1,11077
0,11077


## Save to .parquet file

In [29]:
gene_w_label.write_parquet(os.path.join(WORKING_DIR, 'dataset', f'gene_w_label_{label_column}.parquet'))

In [30]:
test_pl = pl.read_parquet(os.path.join(WORKING_DIR, 'dataset', f'gene_w_label_{label_column}.parquet'))

In [31]:
test_pl

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1,label
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,f64,i32
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0888452,0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,4.04743,1
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0,26.7934,1
…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
